# Simple: Currency Field Analyzer with PAMOLA.CORE

**Goal**: Learn currency field analysis basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze currency fields with locale-aware parsing
- Detect multiple currencies and outliers
- Understand distribution characteristics and normality

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 01_currency_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.currency import (
        CurrencyOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from examples/data_examples/sample.csv
- Auto-creates sample data if file doesn't exist
- Review dataset structure and statistics

**What you'll see:**
- File path confirmation or creation message
- Dataset summary (200 records, 6 columns)
- First 5 records preview
- Column statistics (types, unique values, ranges)
- Currency field with mixed formats: $1,234.56, €1.234,56, £1,234.56

**Note:** Sample includes multiple currencies (USD, EUR, GBP) and some invalid entries for demonstration

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample transaction data with various currency formats
    np.random.seed(42)
    n_samples = 200
    
    # Generate realistic currency amounts
    amounts_usd = np.random.lognormal(mean=6.0, sigma=1.5, size=n_samples//2).round(2)
    amounts_eur = np.random.lognormal(mean=5.8, sigma=1.4, size=n_samples//4).round(2)
    amounts_gbp = np.random.lognormal(mean=5.9, sigma=1.3, size=n_samples//4).round(2)
    
    # Format currency values with symbols
    currency_values = []
    
    # USD format with $ symbol
    for amt in amounts_usd:
        currency_values.append(f"${amt:,.2f}")
    
    # EUR format with € symbol
    for amt in amounts_eur:
        # European format: comma for decimal, period for thousands
        formatted = f"{amt:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
        currency_values.append(f"€{formatted}")
    
    # GBP format with £ symbol
    for amt in amounts_gbp:
        currency_values.append(f"£{amt:,.2f}")
    
    # Add some invalid/null entries
    currency_values[5] = None
    currency_values[15] = "N/A"
    currency_values[25] = "Free"
    currency_values[35] = "$-50.00"  # Negative value
    
    sample_data = pd.DataFrame({
        'transaction_id': [f'TXN{i:05d}' for i in range(1, n_samples + 1)],
        'amount': currency_values,
        'merchant_category': np.random.choice(['Retail', 'Food', 'Entertainment', 'Travel', 'Healthcare'], n_samples),
        'payment_method': np.random.choice(['Credit Card', 'Debit Card', 'Cash', 'Digital Wallet'], n_samples),
        'transaction_date': pd.date_range('2024-01-01', periods=n_samples, freq='D'),
        'customer_id': [f'C{i:04d}' for i in np.random.randint(1, 100, n_samples)]
    })
    
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): {len(df[col])} records")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name**: Edit `create_config_kwargs()` function
   - Change `"field_name": "current_salary_cad"` to YOUR currency column
   - Available currency field: amount
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (✓ Field exists with data type and sample values)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs confirmed (✓)
- DataSource created (✓)
- Progress tracker ready (✓)

**Note:** Default field "current_salary_cad" doesn't exist - you MUST customize to "amount"!

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "amount"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "current_salary_cad"  # Field to analyze - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists in dataset
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to analyze: '{field_name}'")
print(f"  Data type: {df[field_name].dtype}")
print(f"  Non-null count: {df[field_name].notna().sum()}")
print(f"  Sample values:")
for val in df[field_name].dropna().head(5):
    print(f"    • {val}")

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="currency_001",
    task_type="currency_analysis",
    description=f"Currency analysis of '{field_name}' field.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field_name for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Analyzing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review operation configuration
- Run to execute currency analysis
- Monitor progress bar (6 processing steps)

**What you'll see:**
- Configuration summary (field, locale, bins, outlier/normality settings)
- Progress bar: parse → validate → analyze → detect outliers → test normality → save
- Completion message with status and artifact count (3-5 files expected)
- Status = "success" (verify no errors)

**Key parameters:**
- `field_name`: Currency field to analyze (from config)
- `locale='en_US'`: Format parser (en_US, fr_FR, de_DE)
- `detect_outliers=True`: Statistical outlier detection
- `test_normality=True`: Shapiro-Wilk normality test
- `generate_visualization=True`: Create histogram, boxplot, Q-Q plot

In [ ]:
# Create operation with production-style configuration
operation = CurrencyOperation(
    # Core parameters
    field_name=field_name,               # From config
    locale='en_US',                      # Locale for parsing
    bins=10,                             # Histogram bins
    
    # Analysis parameters
    detect_outliers=True,                # Detect outliers
    test_normality=True,                 # Test normality
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,             # Disable for small data
    parallel_processes=1,
    use_dask=False,
    chunk_size=10000,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,         # Create visualization artifacts
    save_output=True,                    # Save results to files
    force_recalculation=False            # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Locale:           {operation.locale}")
print(f"  Bins:             {operation.bins}")
print(f"  Detect outliers:  {operation.detect_outliers}")
print(f"  Test normality:   {operation.test_normality}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing currency analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and display currency analysis results
- Review multi-currency detection and statistics
- Check for outliers and normality

**What you'll see:**
- Generated artifacts list (statistics JSON, samples, visualizations)
- Basic information (total/valid/null/invalid counts)
- Currency distribution (USD, EUR, GBP with counts and percentages)
- Statistical summary (min, max, mean, median, std, skewness, kurtosis)
- Outlier detection results (count, bounds, percentage)
- Normality test results (Shapiro-Wilk statistic, p-value)

**Note:** Multi-currency flag indicates mixed currencies. Outliers use IQR method. P-value < 0.05 suggests non-normal distribution.

**Generated artifacts:**
- Statistics JSON in output/
- Sample records CSV in dictionaries/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load statistics file
stats_files = list(task_dir.glob('output/*_stats.json'))
if stats_files:
    stats_file = stats_files[0]
    
    with open(stats_file, 'r') as f:
        stats_data = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 CURRENCY ANALYSIS RESULTS")
    print("=" * 80)
    
    # Display basic information
    print("\n🔍 Basic Information:")
    print("-" * 60)
    print(f"  Field name: {stats_data.get('field_name', 'N/A')}")
    print(f"  Total rows: {stats_data.get('total_rows', 0):,}")
    print(f"  Valid values: {stats_data.get('valid_count', 0):,}")
    print(f"  Null values: {stats_data.get('null_count', 0):,} ({stats_data.get('null_percentage', 0):.2f}%)")
    print(f"  Invalid values: {stats_data.get('invalid_count', 0):,} ({stats_data.get('invalid_percentage', 0):.2f}%)")
    print(f"  Locale used: {stats_data.get('locale_used', 'N/A')}")
    
    # Display currency information
    if 'currency_counts' in stats_data:
        currency_counts = stats_data['currency_counts']
        multi_currency = stats_data.get('multi_currency', False)
        
        print("\n💱 Currency Information:")
        print("-" * 60)
        print(f"  Multi-currency: {'Yes' if multi_currency else 'No'}")
        print(f"  Currencies detected: {len(currency_counts)}")
        if currency_counts:
            print(f"\n  Currency distribution:")
            for currency, count in sorted(currency_counts.items(), key=lambda x: x[1], reverse=True):
                pct = (count / stats_data.get('valid_count', 1)) * 100
                bar = '█' * int(pct / 5)
                print(f"    {currency:5s} {bar:20s} {count:4d} ({pct:5.1f}%)")
    
    # Display statistics
    if 'stats' in stats_data:
        stats = stats_data['stats']
        
        print("\n📈 Statistical Summary:")
        print("-" * 60)
        print(f"  Minimum:    {stats.get('min', 0):>12.2f}")
        print(f"  Maximum:    {stats.get('max', 0):>12.2f}")
        print(f"  Mean:       {stats.get('mean', 0):>12.2f}")
        print(f"  Median:     {stats.get('median', 0):>12.2f}")
        print(f"  Std Dev:    {stats.get('std', 0):>12.2f}")
        print(f"  Skewness:   {stats.get('skewness', 0):>12.4f}")
        print(f"  Kurtosis:   {stats.get('kurtosis', 0):>12.4f}")
        
        # Display zero and negative counts
        print("\n📊 Value Distribution:")
        print("-" * 60)
        print(f"  Zero values: {stats.get('zero_count', 0)} ({stats.get('zero_percentage', 0):.2f}%)")
        print(f"  Negative values: {stats.get('negative_count', 0)} ({stats.get('negative_percentage', 0):.2f}%)")
        
        # Display outliers if detected
        if 'outliers' in stats and stats['outliers'].get('count', 0) > 0:
            outliers = stats['outliers']
            print("\n⚠️  Outlier Detection:")
            print("-" * 60)
            print(f"  Outliers found: {outliers.get('count', 0)} ({outliers.get('percentage', 0):.2f}%)")
            if 'lower_bound' in outliers:
                print(f"  Lower bound: {outliers.get('lower_bound', 0):.2f}")
            if 'upper_bound' in outliers:
                print(f"  Upper bound: {outliers.get('upper_bound', 0):.2f}")
        
        # Display normality test results
        if 'normality' in stats:
            normality = stats['normality']
            print("\n📐 Normality Testing:")
            print("-" * 60)
            print(f"  Is normal: {normality.get('is_normal', False)}")
            if 'shapiro' in normality:
                shapiro = normality['shapiro']
                print(f"  Shapiro-Wilk statistic: {shapiro.get('statistic', 0):.6f}")
                print(f"  P-value: {shapiro.get('p_value', 0):.6f}")
        
        # Display semantic notes if present
        if 'semantic_notes' in stats and stats['semantic_notes']:
            print("\n💡 Semantic Notes:")
            print("-" * 60)
            for note in stats['semantic_notes']:
                print(f"  • {note}")
    
    # Display result metrics
    if result.metrics:
        print("\n" + "=" * 80)
        print("📊 SUMMARY METRICS")
        print("=" * 80)
        for key, value in list(result.metrics.items())[:15]:
            if isinstance(value, dict):
                print(f"  {key}:")
                for k, v in list(value.items())[:5]:
                    if isinstance(v, float):
                        print(f"    {k:20s}: {v:.4f}")
                    else:
                        print(f"    {k:20s}: {v}")
            elif isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
            else:
                print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Currency analysis provides locale-aware parsing and multi-currency detection!")
else:
    print("⚠️  No statistics file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files in directory structure
- Navigate to task_dir for manual inspection
- Verify file counts and sizes

**What you'll see:**
- Directory structure with file counts per folder
- File names with sizes (KB) for up to 5 files per folder
- Full path to task directory for external access
- Total artifacts confirmation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Statistics JSON
├── dictionaries/     # Sample records CSV
├── visualizations/   # PNG charts (histogram, boxplot, Q-Q plot)
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'dictionaries', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:  # Show first 5 files
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance metrics (if available)
2. **Statistics JSON**: Detailed currency analysis with all metrics
3. **Sample Records**: Representative currency values
4. **Visualizations**: Distribution histogram, boxplot, Q-Q plot

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            print("\n📊 KEY METRICS:")
            metric_keys = [
                ('total_rows', 'Total Rows'),
                ('valid_count', 'Valid Count'),
                ('null_count', 'Null Count'),
                ('null_percentage', 'Null Percentage'),
                ('multi_currency', 'Multi-Currency'),
                ('currency_count', 'Currency Count')
            ]
            
            for key, label in metric_keys:
                if key in metrics:
                    value = metrics[key]
                    if isinstance(value, float):
                        print(f"   {label}: {value:.4f}")
                    else:
                        print(f"   {label}: {value}")
            
            # Display nested statistics
            if 'statistics' in metrics:
                print("\n📈 STATISTICS:")
                stats = metrics['statistics']
                for key, value in list(stats.items())[:10]:
                    if value is not None:
                        if isinstance(value, float):
                            print(f"   {key}: {value:.4f}")
                        else:
                            print(f"   {key}: {value}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. STATISTICS JSON (NEWEST)
print("\n\n2️⃣ DETAILED STATISTICS:")
print("-" * 80)

output_dir = task_dir / "output"

if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")

else:
    stats_files = list(output_dir.glob("*_stats.json"))

    if not stats_files:
        print("⚠️  No statistics files found")

    else:
        # Pick newest file
        stats_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_stats_file = stats_files[0]

        mtime = datetime.fromtimestamp(latest_stats_file.stat().st_mtime)
        print(f"✓ Found {len(stats_files)} statistics file(s)")
        print(f"📄 Loading LATEST: {latest_stats_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_stats_file.stat().st_size / 1024:.1f} KB")
        print("=" * 80)

        try:
            with open(latest_stats_file, "r", encoding="utf-8") as f:
                full_stats = json.load(f)

            # BASIC INFO
            print("\n📊 BASIC INFORMATION:")
            print("-" * 60)

            field_name = full_stats.get("field_name", "N/A")
            total_rows = full_stats.get("total_rows", 0)
            valid_count = full_stats.get("valid_count", 0)
            null_count = full_stats.get("null_count", 0)
            invalid_count = full_stats.get("invalid_count", 0)

            print(f"  Field Name : {field_name}")
            print(f"  Total Rows: {total_rows:,}")
            print(f"  Valid     : {valid_count:,}")
            print(f"  Null      : {null_count:,}")
            print(f"  Invalid   : {invalid_count:,}")

            if "null_percentage" in full_stats:
                print(f"  Null %    : {full_stats['null_percentage']:.2f}%")

            # CURRENCY INFO (IF ANY)
            if full_stats.get("currency_counts"):
                print("\n💱 CURRENCY INFORMATION:")
                print("-" * 60)
                for cur, cnt in full_stats["currency_counts"].items():
                    print(f"  {cur}: {cnt}")

                print(f"  Multi-currency: {full_stats.get('multi_currency', False)}")

            # NUMERIC STATISTICS (ONLY IF PRESENT)
            stats = full_stats.get("stats")

            is_numeric = isinstance(stats, dict) and any(
                k in stats for k in ("min", "max", "mean", "median")
            )

            if is_numeric:
                print("\n📈 NUMERIC STATISTICS:")
                print("-" * 60)

                print(f"  Min    : {stats.get('min')}")
                print(f"  Max    : {stats.get('max')}")
                print(f"  Mean   : {stats.get('mean')}")
                print(f"  Median : {stats.get('median')}")
                print(f"  Std    : {stats.get('std')}")

                print(f"\n  Zero Count     : {stats.get('zero_count', 0)}")
                print(f"  Negative Count: {stats.get('negative_count', 0)}")

                # PERCENTILES
                if "percentiles" in stats:
                    print("\n📊 PERCENTILES:")
                    for p, v in stats["percentiles"].items():
                        print(f"  {p:>6}: {v}")

                # HISTOGRAM
                if "histogram" in stats:
                    hist = stats["histogram"]
                    print("\n📊 HISTOGRAM:")
                    for b, c in zip(hist.get("bins", []), hist.get("counts", [])):
                        print(f"  {b:<25} : {c}")

                # OUTLIERS
                if "outliers" in stats:
                    o = stats["outliers"]
                    print("\n🚨 OUTLIERS:")
                    print(f"  Method       : IQR")
                    print(f"  Lower Bound  : {o.get('lower_bound')}")
                    print(f"  Upper Bound  : {o.get('upper_bound')}")
                    print(f"  Count        : {o.get('count', 0)} ({o.get('percentage', 0)}%)")

                # NORMALITY TESTS
                if "normality" in stats:
                    n = stats["normality"]
                    print("\n🧪 NORMALITY TESTS:")
                    print(f"  Is Normal        : {n.get('is_normal', False)}")
                    print(f"  Tests Passed     : {n.get('normal_test_passed', 0)}/{n.get('normal_test_count', 0)}")

                    if "shapiro" in n:
                        print(f"  Shapiro p-value  : {n['shapiro'].get('p_value')}")

                    if "ks" in n:
                        print(f"  KS p-value       : {n['ks'].get('p_value')}")

            else:
                # NON-NUMERIC FIELD
                print("\n🔤 NON-NUMERIC FIELD:")
                print("-" * 60)
                print("  This field is not numeric.")
                print("  Min / Max / Mean / Std are not applicable.")

            print("\n✅ Detailed statistics rendered successfully.")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading statistics: {e}")

# 3. SAMPLE RECORDS (NEWEST)
print("\n\n3️⃣ SAMPLE RECORDS:")
print("-" * 80)

dict_dir = task_dir / 'dictionaries'
if not dict_dir.exists():
    print(f"⚠️  Dictionaries directory not found: {dict_dir}")
else:
    sample_files = list(dict_dir.glob('*_sample.csv'))
    
    if not sample_files:
        print("   ℹ️  No sample files found")
    else:
        sample_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_sample_file = sample_files[0]
        
        mtime = datetime.fromtimestamp(latest_sample_file.stat().st_mtime)
        print(f"✓ Found {len(sample_files)} sample file(s)")
        print(f"📄 Loading LATEST: {latest_sample_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_sample_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            sample_df = pd.read_csv(latest_sample_file)
            print(f"📊 Shape: {sample_df.shape[0]} rows × {sample_df.shape[1]} columns\n")
            print(sample_df.head(20).to_string(index=False))
            
            if len(sample_df) > 20:
                print(f"\n... and {len(sample_df) - 20} more records")
        
        except Exception as e:
            print(f"❌ Error reading sample: {e}")

# 4. VISUALIZATIONS (NEWEST BATCH)
print("\n\n4️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            for i, viz_file in enumerate(latest_viz_batch, 1):
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with locale config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review currency statistics and multi-currency detection  

**Key takeaways:**
- **Locale-aware parsing**: Handles different currency formats ($ vs € vs £)
- **Multi-currency detection**: Automatically identifies and counts currencies
- **Outlier detection**: Statistical identification of unusual values
- **Normality testing**: Shapiro-Wilk test for distribution analysis
- **Comprehensive statistics**: Min, max, mean, median, std, skewness, kurtosis
- **Special value tracking**: Zero and negative value counts
- **Semantic notes**: Automatic insights about data characteristics
- All analysis saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_currency_analyzer_advanced.ipynb`** includes:
- Multi-locale support and custom locale definitions
- Currency conversion with exchange rates
- Large dataset processing with Dask
- Parallel processing for performance
- Advanced outlier detection methods
- Custom binning strategies for histograms

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Currency Analysis Guide](../../docs/currency_analysis.md)
- 📋 [Locale Reference](../../docs/locale_reference.md)